# 01 — Eksplorasi, Cleaning, QC, dan Ekspor Data Pasut Cilacap 2025

**Tujuan notebook:** memastikan workbook pasang-surut 2025 sudah benar secara struktur dan kualitas sebelum dipakai untuk database, dashboard, Admiralty, dan Least Square.

> Catatan penting: workbook Excel asli tidak diubah. Notebook hanya membaca file raw dan menghasilkan file processed baru.

## Alur kerja

`Excel raw → baca 12 sheet bulan → ambil TGL + JAM 1..24 → hapus hari yang seluruhnya kosong → ubah wide ke long → buat timestamp WIB → QC → ekspor CSV`

Hasil akhir yang ditargetkan adalah **8.760 observasi** (365 hari × 24 jam).

In [ ]:
# CELL 1 — Import library dasar dan menentukan lokasi project
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('PROJECT_ROOT =', PROJECT_ROOT)


In [ ]:
# CELL 2 — Import modul project yang sudah kita buat
from src.data_loader import RAW_FILE, MONTH_SHEETS, load_month
from src.cleaning import clean_month
from src.preprocessing import build_annual_dataset
from src.qc import monthly_qc, annual_qc, validate_annual_dataset

print('File raw :', RAW_FILE)
print('File ada :', RAW_FILE.exists())
print('Sheet bulan:', MONTH_SHEETS)


## 1. Periksa workbook dan daftar sheet

Workbook juga memiliki sheet grafik/tabel bantu. Untuk dataset utama kita hanya menggunakan 12 sheet bulan: JAN sampai DES.

In [ ]:
# CELL 3 — Melihat seluruh nama sheet pada workbook
xls = pd.ExcelFile(RAW_FILE)
print('Semua sheet pada workbook:')
for i, sheet in enumerate(xls.sheet_names, start=1):
    print(f'{i:02d}. {sheet}')


## 2. QC per bulan

Di sini kita mengecek jumlah hari, jumlah observasi, data kosong, dan apakah setiap hari mempunyai 24 bacaan.

In [ ]:
# CELL 4 — Membuat tabel QC untuk 12 bulan
df_monthly_qc = monthly_qc(load_month)
display(df_monthly_qc)


In [ ]:
# CELL 5 — Ringkasan QC bulanan
print('Total hari valid       :', df_monthly_qc['jumlah_hari'].sum())
print('Total observasi        :', df_monthly_qc['jumlah_observasi'].sum())
print('Total missing observasi:', df_monthly_qc['missing_observasi'].sum())
print('Hari lengkap           :', df_monthly_qc['hari_lengkap'].sum())


## 3. Membentuk dataset tahunan long-format

Format Excel berbentuk **wide**: satu baris = satu tanggal dan 24 kolom = JAM 1–24.

Untuk database dan dashboard, kita ubah menjadi **long**: satu baris = satu observasi pasut.

In [ ]:
# CELL 6 — Bersihkan semua sheet bulan dan gabungkan menjadi dataset tahunan
monthly_clean = {month: clean_month(load_month(month)) for month in MONTH_SHEETS}
df_annual = build_annual_dataset(monthly_clean, year=2025)

print('Ukuran dataset:', df_annual.shape)
display(df_annual.head(10))
display(df_annual.tail(10))


## 4. Cek aturan timestamp

Pada workbook, JAM 1–23 berarti 01:00–23:00. JAM 24 diperlakukan sebagai 00:00 pada hari berikutnya. Jadi JAM 24 tanggal 31 Desember 2025 menjadi `2026-01-01 00:00 WIB`. Ini adalah representasi waktu, bukan berarti data berubah menjadi data observasi tahun 2026.

In [ ]:
# CELL 7 — Melihat contoh timestamp awal dan akhir
display(df_annual[['timestamp','tahun_data','bulan','tanggal','jam','tinggi_m']].head(30))
display(df_annual[['timestamp','tahun_data','bulan','tanggal','jam','tinggi_m']].tail(10))


In [ ]:
# CELL 8 — QC dataset tahunan
qc = annual_qc(df_annual)
for key, value in qc.items():
    print(f'{key}: {value}')


In [ ]:
# CELL 9 — Validasi ketat sebelum ekspor
# Jika ada masalah struktural, baris berikut akan menghentikan proses ekspor.
validate_annual_dataset(df_annual, strict=True)
print('QC LULUS — dataset memenuhi pemeriksaan dasar.')


## 5. Pemeriksaan duplicate dan kontinuitas waktu

In [ ]:
# CELL 10 — Memeriksa timestamp duplikat
dupes = df_annual[df_annual['timestamp'].duplicated(keep=False)]
print('Jumlah baris yang terlibat duplicate timestamp:', len(dupes))
display(dupes.head(20))


In [ ]:
# CELL 11 — Menampilkan selisih waktu antarobservasi
interval = df_annual['timestamp'].diff().dropna().dt.total_seconds() / 3600
print('Interval minimum (jam):', interval.min())
print('Interval maksimum (jam):', interval.max())
print('Jumlah interval bukan 1 jam:', (interval != 1).sum())


## 6. Statistik dasar dan grafik

Bagian ini bukan perhitungan Admiralty/Least Square. Tujuannya hanya memahami bentuk data observasi sebelum analisis harmonik.

In [ ]:
# CELL 12 — Statistik dasar tinggi muka air
display(df_annual['tinggi_m'].describe())


In [ ]:
# CELL 13 — Grafik seluruh observasi pasut tahun 2025
fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(df_annual['timestamp'], df_annual['tinggi_m'], linewidth=0.7)
ax.set_title('Data Pasang Surut Cilacap 2025')
ax.set_xlabel('Waktu (WIB)')
ax.set_ylabel('Tinggi muka air (m)')
ax.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()


## 7. Ekspor CSV siap database

File ini adalah hasil preprocessing, bukan file raw. Struktur satu baris = satu observasi pasut.

In [ ]:
# CELL 14 — Menyimpan dataset final ke CSV
processed_dir = PROJECT_ROOT / 'data' / 'processed'
processed_dir.mkdir(parents=True, exist_ok=True)

output_csv = processed_dir / 'pasut_2025_long.csv'
df_annual.to_csv(output_csv, index=False, encoding='utf-8-sig')

print('CSV berhasil dibuat:')
print(output_csv)
print('Jumlah baris:', len(df_annual))


In [ ]:
# CELL 15 — Baca ulang CSV untuk memastikan file benar-benar dapat dibaca kembali
df_check = pd.read_csv(output_csv)
print('Jumlah baris setelah dibaca ulang:', len(df_check))
print('Kolom:', df_check.columns.tolist())
display(df_check.head())


## 8. Keputusan tahap data

Jika CELL 9 menyatakan **QC LULUS**, maka dataset `pasut_2025_long.csv` sudah dapat dijadikan sumber data untuk database/dashboard.

**Jangan menghapus data 2025 hanya karena Admiralty membutuhkan 29 hari.** Data 8.760 baris ini menjadi master dataset. Nanti kita membuat subset 29 hari khusus untuk perbandingan Admiralty vs Least Square.